# Chapter 13: Affine Geometry

_Source span inspected: Coxeter, Introduction to Geometry, Chapter 13, printed pages 191-228; PDF pages 209-246. The scanned source pages were rendered privately for orientation only. This notebook uses original prose, code, and generated diagrams rather than source page images or copied figures._

## Chapter Goal

Affine geometry keeps incidence, betweenness, parallelism, ratios on parallel lines, and signed area/volume ratios, while deliberately forgetting Euclidean lengths and angles. By the end of this notebook you should be able to recognize which statements survive an arbitrary affinity, which require an equiaffinity, and how vectors, centroids, barycentric coordinates, and lattices are all different languages for the same affine structure.

## Computational Translation Guide

| Book idea | Computational model | What to inspect | Invariant check |
| --- | --- | --- | --- |
| Parallel lines | Direction vectors with zero 2D cross product | Lines remain in one parallel pencil | `cross(u, v) = 0` |
| Desargues-style affine axiom | Triangles related by a central dilation or translation | Side pairs parallel; connector lines concurrent or parallel | concurrence and side-direction residuals |
| Dilatation | `x -> O + k(x - O)` or `x -> x + t` | Image segment is parallel to original | determinant scale and line-pair tests |
| Affinity | `x -> A x + b`, `det(A) != 0` | Lines, collinearity, and ratios on parallel lines survive | rank, determinant, area scale |
| Equiaffinity | Affinity with `det(A) = 1` | Signed area is unchanged while shape changes | transformed area/original area equals 1 |
| Lattice | Integer span of a basis | Visible points, unit cells, Pick/Farey structure | gcd and determinant conditions |
| Vectors and centroids | Difference vectors and weighted averages | The centroid is independent of temporary origin | weighted residual is zero |
| Barycentric coordinates | Homogeneous masses at vertices | Lines become linear homogeneous equations | determinant line equation; Ceva/Menelaus products |
| Affine space and 3D lattices | Basis vectors in `R^3`, integer coordinate triples | Parallelepiped cells and rational planes | volume determinant and unimodular basis change |


## Visual Storyboard Implemented

| Storyboard item | Artifact | Learner inspection target | Validation |
| --- | --- | --- | --- |
| Parallelism and Desargues axiom | `parallelism-desargues-configuration.svg`, `desargues-proof-dependency.svg` | Side pairs parallel; connector lines concurrent or parallel | cross-product and graph DAG checks |
| Dilatations, translations, half-turns | `dilatation-translation-halfturns.svg` | One invariant point versus no invariant point | determinant and composition checks |
| Affinities and equiaffinities | `affinity-equiaffinity-area.svg`, `affine-map-lab.html` | Area changes by `det(A)`; determinant-one maps preserve it | numeric area ratios and SymPy identities |
| 2D lattices | `lattice-pick-farey-visibility.svg` | Boundary/interior lattice points and adjacent fractions | Pick residual and Farey determinant |
| Vectors, centroids, barycentric coordinates | `vectors-centroids-barycentric.svg` | Weighted average equals area decomposition | origin independence, reconstruction, Ceva/Menelaus |
| 3D affine space and lattices | `three-dimensional-lattice-rational-planes.html` | Unit cell volume, unimodular basis change, first rational planes | volume, gcd, and plane residual checks |


In [1]:

from __future__ import annotations
import json, math, sys
from fractions import Fraction
from itertools import product
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display

CHAPTER_NO=13
HERE=Path.cwd().resolve(); BOOK_ROOT=None
for candidate in [HERE,*HERE.parents]:
    if (candidate/"00-book-index.ipynb").exists() and (candidate/"utils").exists():
        BOOK_ROOT=candidate; break
if BOOK_ROOT is None: raise RuntimeError("Could not locate Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path: sys.path.insert(0,str(BOOK_ROOT))
from utils.artifacts import artifact_path,ensure_chapter_artifact_dirs,display_artifact,assert_artifacts,write_json,write_csv,book_relative
artifact_dirs=ensure_chapter_artifact_dirs(CHAPTER_NO,BOOK_ROOT)
FIG_DIR=artifact_dirs["figures"]; HTML_DIR=artifact_dirs["html"]; CHECK_DIR=artifact_dirs["checks"]; TABLE_DIR=artifact_dirs["tables"]; DATA_DIR=artifact_dirs["data"]
for stale_name in ["concept_configuration.svg","parameter_experiment.svg"]:
    stale_path=FIG_DIR/stale_name
    if stale_path.exists(): stale_path.unlink()
plt.rcParams.update({"figure.dpi":130,"savefig.dpi":160,"font.size":10,"axes.titlesize":11,"axes.labelsize":9})
ARTIFACTS_WRITTEN=[]; CHECK_FILES=[]
def remember(path:Path,concept:str,kind:str)->Path:
    ARTIFACTS_WRITTEN.append({"concept":concept,"kind":kind,"path":book_relative(path,BOOK_ROOT)}); return path
def write_check(filename:str,payload:dict)->Path:
    path=write_json(CHECK_DIR/filename,payload); CHECK_FILES.append(path); remember(path,payload.get("concept",filename.removesuffix(".json")),"checks"); return path
def cross2(u,v):
    u=np.asarray(u); v=np.asarray(v); return float(u[0]*v[1]-u[1]*v[0])
def signed_area(poly):
    poly=np.asarray(poly,float); x=poly[:,0]; y=poly[:,1]; return 0.5*float(np.dot(x,np.roll(y,-1))-np.dot(y,np.roll(x,-1)))
def line_intersection(p1,p2,q1,q2):
    r=p2-p1; s=q2-q1; denom=cross2(r,s)
    if abs(denom)<1e-12: raise ValueError("parallel lines do not have a finite intersection")
    return p1+(cross2(q1-p1,s)/denom)*r
def draw_arrow(ax,start,vec,color="#34495e",label=None,lw=1.8):
    ax.arrow(start[0],start[1],vec[0],vec[1],length_includes_head=True,head_width=.06,head_length=.10,fc=color,ec=color,lw=lw)
    if label:
        end=np.asarray(start)+np.asarray(vec); ax.text(end[0]+.05,end[1]+.05,label,color=color,weight="bold")
def label_points(ax,pts,offset=(.05,.05)):
    for name,pt in pts.items(): ax.plot(pt[0],pt[1],"o",ms=4,color="#17202a"); ax.text(pt[0]+offset[0],pt[1]+offset[1],name,fontsize=9)
def set_equal_2d(ax):
    ax.set_aspect("equal",adjustable="box"); ax.grid(True,color="#e5e7eb",lw=.7); ax.tick_params(labelbottom=False,labelleft=False,length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
def affine(A,pts,b=np.zeros(2)):
    pts=np.asarray(pts,float); return pts@np.asarray(A,float).T+np.asarray(b,float)
source_span={"book":"Introduction to Geometry","chapter":13,"printed_pages":"191-228","pdf_pages":"209-246","source_map":"pdf_page = printed_page + 18","inspected":True}
write_check("source-span.json",{"concept":"source span orientation",**source_span})
source_span


{'book': 'Introduction to Geometry',
 'chapter': 13,
 'printed_pages': '191-228',
 'pdf_pages': '209-246',
 'source_map': 'pdf_page = printed_page + 18',
 'inspected': True}

## 1. Parallelism and the Desargues Axiom

The chapter begins by turning the parallel postulate into an affine axiom: through a point not on a line there is exactly one parallel to the line. That makes parallelism an equivalence relation on lines. The extra Desargues-style axiom supplies a way to control triangles whose corresponding side pairs are parallel.

The left panel below shows the central case: a triangle is sent to a larger triangle by a dilation from `O`, so corresponding sides are parallel and the three connector lines meet at `O`. The right panel shows the translation case: corresponding sides are still parallel, but the connector lines now form a parallel pencil rather than meeting at a finite point.

In [2]:

O=np.array([-.25,3.05])
ABC={"A":np.array([.15,1.75]),"B":np.array([-1.35,.10]),"C":np.array([1.25,.05])}
scale=1.75
ABCp={name+"'":O+scale*(pt-O) for name,pt in ABC.items()}
translation=np.array([.95,-2.45])
ABCt={name+"''":pt+translation for name,pt in ABC.items()}
fig,axes=plt.subplots(1,2,figsize=(12.4,5.4))
for ax,title in zip(axes,["central dilation: connectors concur","translation: connectors are parallel"]): set_equal_2d(ax); ax.set_title(title)
ax=axes[0]; tri1=np.array([ABC["A"],ABC["B"],ABC["C"]]); tri2=np.array([ABCp["A'"],ABCp["B'"],ABCp["C'"]])
ax.add_patch(Polygon(tri1,closed=True,fill=False,lw=2,ec="#2563eb")); ax.add_patch(Polygon(tri2,closed=True,fill=False,lw=2,ec="#dc2626"))
for name,pt in ABC.items():
    target=ABCp[name+"'"]; ax.plot([O[0],target[0]],[O[1],target[1]],color="#6b7280",lw=1,ls="--")
label_points(ax,{**ABC,**ABCp,"O":O}); ax.set_xlim(-2.3,2.4); ax.set_ylim(-1.8,3.5)
ax=axes[1]; tri3=np.array([ABCt["A''"],ABCt["B''"],ABCt["C''"]])
ax.add_patch(Polygon(tri1,closed=True,fill=False,lw=2,ec="#2563eb")); ax.add_patch(Polygon(tri3,closed=True,fill=False,lw=2,ec="#059669"))
for name,pt in ABC.items():
    target=ABCt[name+"''"]; ax.plot([pt[0],target[0]],[pt[1],target[1]],color="#6b7280",lw=1.2,ls="--")
label_points(ax,{**ABC,**ABCt}); ax.set_xlim(-2,3); ax.set_ylim(-3.1,2.2)
fig.suptitle("Desargues affine alternatives: finite center or direction at infinity",y=1.02,fontsize=13); fig.tight_layout()
parallel_path=remember(FIG_DIR/"parallelism-desargues-configuration.svg","parallelism and Desargues axiom","figures")
fig.savefig(parallel_path,bbox_inches="tight"); plt.close(fig)
side_pairs=[("A","B"),("B","C"),("C","A")]
central_side_residuals={f"{u}{v}":cross2(ABC[v]-ABC[u],ABCp[v+"'"]-ABCp[u+"'"]) for u,v in side_pairs}
translation_side_residuals={f"{u}{v}":cross2(ABC[v]-ABC[u],ABCt[v+"''"]-ABCt[u+"''"]) for u,v in side_pairs}
intersection_ab=line_intersection(ABC["A"],ABCp["A'"],ABC["B"],ABCp["B'"])
intersection_ac=line_intersection(ABC["A"],ABCp["A'"],ABC["C"],ABCp["C'"])
translation_connector_residuals={name:float(np.linalg.norm((ABCt[name+"''"]-ABC[name])-translation)) for name in ["A","B","C"]}
checks={"concept":"parallelism and Desargues axiom","central_side_cross_residuals":central_side_residuals,"translation_side_cross_residuals":translation_side_residuals,"central_connector_intersection_error":float(max(np.linalg.norm(intersection_ab-O),np.linalg.norm(intersection_ac-O))),"translation_connector_vector_errors":translation_connector_residuals}
checks["max_abs_residual"]=float(max(max(abs(v) for v in central_side_residuals.values()),max(abs(v) for v in translation_side_residuals.values()),max(translation_connector_residuals.values()),np.linalg.norm(intersection_ab-O),np.linalg.norm(intersection_ac-O)))
assert checks["max_abs_residual"]<1e-10
write_check("parallelism-desargues-checks.json",checks)
display_artifact(parallel_path,width=920)


### Proof Dependency Scaffold

The source develops the chapter in a chain: parallelism plus the Desargues axiom make dilatations well-defined; dilatations make translations and half-turns into an Abelian vector group; affinities act on that group; equiaffinities preserve area; barycentric coordinates turn cevians and transversals into determinant tests; the same vector language then extends to affine space and 3D lattices.

In [3]:

G=nx.DiGraph()
edges=[("13.11 unique parallel","parallel pencils"),("13.12 Desargues axiom","connector theorem"),("parallel pencils","connector theorem"),("connector theorem","unique dilatation from a segment"),("unique dilatation from a segment","central dilatations / translations"),("central dilatations / translations","half-turns and vector group"),("half-turns and vector group","affine coordinates"),("affine coordinates","affinities preserve lines"),("affinities preserve lines","determinant area formula"),("determinant area formula","equiaffinities"),("determinant area formula","Pick and Farey lattice facts"),("half-turns and vector group","centroids"),("centroids","barycentric coordinates"),("barycentric coordinates","Ceva and Menelaus"),("affine coordinates","3D affine space"),("3D affine space","3D lattices and rational planes")]
G.add_edges_from(edges)
layers=[["13.11 unique parallel","13.12 Desargues axiom"],["parallel pencils","connector theorem"],["unique dilatation from a segment","central dilatations / translations"],["half-turns and vector group","affine coordinates"],["affinities preserve lines","determinant area formula","centroids"],["equiaffinities","Pick and Farey lattice facts","barycentric coordinates","3D affine space"],["Ceva and Menelaus","3D lattices and rational planes"]]
subset={i: layer for i,layer in enumerate(layers)}; pos=nx.multipartite_layout(G,subset_key=subset)
fig,ax=plt.subplots(figsize=(12.8,6.6)); ax.set_axis_off()
nx.draw_networkx_edges(G,pos,ax=ax,arrows=True,arrowstyle="-|>",arrowsize=13,edge_color="#6b7280",width=1.2,connectionstyle="arc3,rad=0.04")
nx.draw_networkx_nodes(G,pos,ax=ax,node_color="#eef2ff",edgecolors="#3730a3",node_size=2050,linewidths=1.2)
nx.draw_networkx_labels(G,pos,ax=ax,font_size=8); ax.set_title("Dependency graph for Chapter 13 affine constructions",fontsize=13,pad=18)
proof_path=remember(FIG_DIR/"desargues-proof-dependency.svg","proof dependency scaffold","figures")
fig.savefig(proof_path,bbox_inches="tight"); plt.close(fig)
proof_checks={"concept":"proof dependency scaffold","nodes":G.number_of_nodes(),"edges":G.number_of_edges(),"is_directed_acyclic_graph":nx.is_directed_acyclic_graph(G),"has_path_axioms_to_3d_lattices":nx.has_path(G,"13.12 Desargues axiom","3D lattices and rational planes")}
assert proof_checks["is_directed_acyclic_graph"] and proof_checks["has_path_axioms_to_3d_lattices"]
write_check("desargues-proof-dependency-checks.json",proof_checks)
display_artifact(proof_path,width=940)


## 2. Dilatations, Translations, and Half-Turns

A dilatation sends every line to a parallel line. If the point-pair lines all pass through one point, the dilatation is central. If those point-pair lines are parallel, the map is a translation. Half-turns sit inside the same language: the half-turn interchanging `A` and `B` is the map `x -> A + B - x`, and the product of half-turns `A <-> B` then `B <-> C` is the translation `A -> C`.

In [4]:

center=np.array([-1.6,1.2]); k=1.55
central_pts={"P":np.array([-.9,2.0]),"Q":np.array([.1,1.55]),"R":np.array([-.45,.55])}
central_img={name+"'":center+k*(pt-center) for name,pt in central_pts.items()}
tvec=np.array([1.35,-.75]); T_pts={"A":np.array([1.0,2.1]),"B":np.array([2.45,2.25]),"C":np.array([1.55,1.05])}; T_img={name+"'":pt+tvec for name,pt in T_pts.items()}
A=np.array([-.2,-1.0]); B=np.array([1.2,-.55]); C=np.array([2.55,-1.25]); D=A+C-B; M_ab=.5*(A+B); M_bc=.5*(B+C)
fig,axes=plt.subplots(1,3,figsize=(13.4,4.5))
for ax,title in zip(axes,["central dilatation","translation","two half-turns"]): set_equal_2d(ax); ax.set_title(title)
ax=axes[0]
for name,pt in central_pts.items():
    img=central_img[name+"'"]; ax.plot([center[0],img[0]],[center[1],img[1]],color="#6b7280",ls="--",lw=1); ax.plot([pt[0],img[0]],[pt[1],img[1]],color="#9ca3af",lw=1)
ax.add_patch(Polygon(np.array(list(central_pts.values())),fill=False,ec="#2563eb",lw=2)); ax.add_patch(Polygon(np.array(list(central_img.values())),fill=False,ec="#dc2626",lw=2)); label_points(ax,{**central_pts,**central_img,"O":center}); ax.set_xlim(-2.2,1); ax.set_ylim(-.3,2.8)
ax=axes[1]
ax.add_patch(Polygon(np.array(list(T_pts.values())),fill=False,ec="#2563eb",lw=2)); ax.add_patch(Polygon(np.array(list(T_img.values())),fill=False,ec="#059669",lw=2))
for name,pt in T_pts.items():
    img=T_img[name+"'"]; ax.plot([pt[0],img[0]],[pt[1],img[1]],color="#6b7280",ls="--",lw=1.2)
draw_arrow(ax,np.array([.8,.5]),tvec,color="#059669",label="t"); label_points(ax,{**T_pts,**T_img}); ax.set_xlim(.3,4.3); ax.set_ylim(.1,2.9)
ax=axes[2]; quad=np.array([A,B,C,D]); ax.add_patch(Polygon(quad,closed=True,fill=False,ec="#7c3aed",lw=2)); ax.plot([A[0],B[0]],[A[1],B[1]],color="#374151",lw=1.2); ax.plot([B[0],C[0]],[B[1],C[1]],color="#374151",lw=1.2); ax.plot([A[0],D[0]],[A[1],D[1]],color="#374151",lw=1.2,ls="--")
ax.scatter([M_ab[0],M_bc[0]],[M_ab[1],M_bc[1]],color="#ef4444",zorder=3); ax.text(M_ab[0]-.18,M_ab[1]+.09,"center AB",fontsize=8,color="#ef4444"); ax.text(M_bc[0]-.10,M_bc[1]+.09,"center BC",fontsize=8,color="#ef4444")
draw_arrow(ax,A,C-A,color="#7c3aed",label="A -> C"); label_points(ax,{"A":A,"B":B,"C":C,"D":D}); ax.set_xlim(-.7,3); ax.set_ylim(-1.8,-.1)
fig.suptitle("Dilatations split into finite-center maps, translations, and half-turn products",y=1.02,fontsize=13); fig.tight_layout()
dilatation_path=remember(FIG_DIR/"dilatation-translation-halfturns.svg","dilatations translations half-turns","figures")
fig.savefig(dilatation_path,bbox_inches="tight"); plt.close(fig)
central_segment_residual=cross2(central_pts["Q"]-central_pts["P"],central_img["Q'"]-central_img["P'"])
translation_errors=[float(np.linalg.norm((T_img[name+"'"]-T_pts[name])-tvec)) for name in T_pts]
def H(pair_sum,x): return pair_sum-x
halfturn_product_on_A=H(B+C,H(A+B,A)); halfturn_product_translation=halfturn_product_on_A-A
checks={"concept":"dilatations translations half-turns","central_scale":k,"central_area_scale":k**2,"central_segment_parallel_residual":float(central_segment_residual),"translation_vector_errors":translation_errors,"halfturn_product_vector":halfturn_product_translation.tolist(),"expected_translation_A_to_C":(C-A).tolist(),"halfturn_product_error":float(np.linalg.norm(halfturn_product_translation-(C-A))),"halfturn_square_error":float(np.linalg.norm(H(A+B,H(A+B,np.array([.4,-.2])))-np.array([.4,-.2])))}
assert abs(checks["central_segment_parallel_residual"])<1e-10 and max(translation_errors)<1e-10 and checks["halfturn_product_error"]<1e-10 and checks["halfturn_square_error"]<1e-10
write_check("dilatation-group-checks.json",checks)
display_artifact(dilatation_path,width=960)


## 3. Affinities and Equiaffinities

An affinity is any invertible affine map `x -> A x + b`. It preserves collinearity and parallelism, but it usually changes area by the signed determinant of `A`. An equiaffinity is the special case `det(A) = 1`, so signed areas are preserved even when the picture is stretched or sheared. The diagram below asks the affine question rather than the Euclidean one: which line and area facts remain after shape and angle have changed?

In [5]:

square=np.array([[0,0],[1,0],[1,1],[0,1]]); triangle=np.array([[.1,.1],[.95,.18],[.25,.85]])
grid_lines=[]
for s in np.linspace(0,1,6): grid_lines += [np.array([[s,0],[s,1]]),np.array([[0,s],[1,s]])]
maps=[("identity",np.array([[1.,0.],[0.,1.]]),np.array([0.,0.]),"#2563eb"),("equiaffine shear",np.array([[1.,.75],[0.,1.]]),np.array([1.55,0.]),"#059669"),("hyperbolic stretch",np.array([[1.45,0.],[0.,1/1.45]]),np.array([3.25,0.]),"#7c3aed"),("general affinity",np.array([[1.25,.45],[-.20,.82]]),np.array([5.1,.15]),"#dc2626")]
fig,ax=plt.subplots(figsize=(12.8,4.8)); set_equal_2d(ax)
for title,Amap,bvec,color in maps:
    det=float(np.linalg.det(Amap))
    for line in grid_lines:
        line_t=affine(Amap,line,bvec); ax.plot(line_t[:,0],line_t[:,1],color="#d1d5db",lw=.8)
    sq=affine(Amap,square,bvec); tri=affine(Amap,triangle,bvec)
    ax.add_patch(Polygon(sq,closed=True,fill=False,ec=color,lw=2)); ax.add_patch(Polygon(tri,closed=True,facecolor=color,alpha=.15,edgecolor=color,lw=1.8))
    label=affine(Amap,np.array([[.45,1.16]]),bvec)[0]; ax.text(label[0],label[1],f"{title}\ndet={det:.3f}",ha="center",va="bottom",fontsize=9,color=color)
ax.set_xlim(-.35,7.05); ax.set_ylim(-.6,2.25); ax.set_title("Affinities preserve line structure; determinants measure signed area scale")
fig.tight_layout(); affinity_path=remember(FIG_DIR/"affinity-equiaffinity-area.svg","affinities and equiaffinities","figures"); fig.savefig(affinity_path,bbox_inches="tight"); plt.close(fig)
base_area=signed_area(triangle); area_checks={}
for title,Amap,bvec,_ in maps:
    transformed_area=signed_area(affine(Amap,triangle,bvec)); det=float(np.linalg.det(Amap)); area_checks[title]={"determinant":det,"area_ratio":transformed_area/base_area,"area_scale_error":abs(transformed_area/base_area-det)}
x,y,mu,theta=sp.symbols("x y mu theta",positive=True)
hyperbolic_identity=sp.simplify((mu*x)*(y/mu)-x*y); parabolic_identity=sp.expand((x+1)**2-(2*x+y+1)-(x**2-y)); rot_det=sp.simplify(sp.Matrix([[sp.cos(theta),-sp.sin(theta)],[sp.sin(theta),sp.cos(theta)]]).det())
checks={"concept":"affinities and equiaffinities","area_checks":area_checks,"hyperbolic_xy_invariant_symbolic":str(hyperbolic_identity),"parabolic_affine_invariant_symbolic":str(parabolic_identity),"elliptic_rotation_determinant_symbolic":str(rot_det)}
assert max(v["area_scale_error"] for v in area_checks.values())<1e-12 and hyperbolic_identity==0 and parabolic_identity==0 and rot_det==1
write_check("affinity-area-symbolic-checks.json",checks)
display_artifact(affinity_path,width=960)


## Applied Lab: A One-Parameter Equiaffine Shear

The source treats hyperbolic, parabolic, and elliptic examples of equiaffinities. The lab below isolates a shear family: slide the shear parameter and watch the grid and triangle deform while their signed area stays fixed. The saved HTML artifact keeps the parameter sweep inspectable outside the notebook.

In [6]:

base_triangle=np.array([[-.55,-.35],[.75,-.25],[-.10,.85]]); base_area_lab=signed_area(base_triangle); frame_params=np.linspace(-1.2,1.2,13)
base_grid=[]
for s in np.linspace(-1,1,9): base_grid += [np.array([[s,-1],[s,1]]),np.array([[-1,s],[1,s]])]
def traces_for_param(a):
    Amap=np.array([[1.,a],[0.,1.]]); traces=[]
    for line in base_grid:
        line_t=affine(Amap,line); traces.append(go.Scatter(x=line_t[:,0],y=line_t[:,1],mode="lines",line=dict(color="rgba(107,114,128,0.35)",width=1),showlegend=False,hoverinfo="skip"))
    tri_t=affine(Amap,base_triangle); closed=np.vstack([tri_t,tri_t[0]])
    traces.append(go.Scatter(x=closed[:,0],y=closed[:,1],mode="lines+markers",fill="toself",fillcolor="rgba(5,150,105,0.22)",line=dict(color="#059669",width=3),marker=dict(size=7),name=f"shear a={a:.2f}"))
    return traces
fig=go.Figure(data=traces_for_param(frame_params[0])); fig.frames=[go.Frame(data=traces_for_param(a),name=f"{a:.2f}") for a in frame_params]
fig.update_layout(title="Equiaffine shear family: det(A)=1 and signed area is fixed",width=780,height=620,xaxis=dict(scaleanchor="y",range=[-2.4,2.4],zeroline=True,showgrid=False),yaxis=dict(range=[-1.7,1.7],zeroline=True,showgrid=False),updatemenus=[dict(type="buttons",showactive=False,x=.02,y=1.08,buttons=[dict(label="Play",method="animate",args=[None,{"frame":{"duration":500,"redraw":True},"fromcurrent":True}]),dict(label="Pause",method="animate",args=[[None],{"frame":{"duration":0,"redraw":False},"mode":"immediate"}])])],sliders=[dict(steps=[dict(method="animate",args=[[f"{a:.2f}"],{"mode":"immediate","frame":{"duration":0,"redraw":True}}],label=f"{a:.1f}") for a in frame_params],currentvalue=dict(prefix="shear a = "))])
lab_path=remember(HTML_DIR/"affine-map-lab.html","applied equiaffine shear lab","html"); fig.write_html(lab_path,include_plotlyjs=True,full_html=True)
rows=[]
for a in frame_params:
    Amap=np.array([[1.,a],[0.,1.]]); tri_t=affine(Amap,base_triangle); rows.append({"shear_parameter":float(a),"determinant":float(np.linalg.det(Amap)),"signed_area":signed_area(tri_t),"area_error":abs(signed_area(tri_t)-base_area_lab)})
lab_csv=write_csv(DATA_DIR/"equiaffine-shear-lab.csv",rows); remember(lab_csv,"applied equiaffine shear lab","data")
checks={"concept":"applied equiaffine shear lab","parameters":[row["shear_parameter"] for row in rows],"max_determinant_error":float(max(abs(row["determinant"]-1.) for row in rows)),"max_area_error":float(max(row["area_error"] for row in rows)),"base_signed_area":float(base_area_lab)}
assert checks["max_determinant_error"]<1e-12 and checks["max_area_error"]<1e-12
write_check("equiaffine-shear-lab-checks.json",checks)
display_artifact(lab_path,width=820)


## 4. Two-Dimensional Lattices: Unit Cells, Pick, and Farey

An affine lattice is the set of integer combinations of two independent translations. In Euclidean geometry the shape of a lattice cell matters. In affine geometry only its signed area and integer structure matter. Visible lattice points are the primitive integer pairs, and Pick's theorem turns a polygon with lattice vertices into an area count. The right panel shows the Farey sequence `F_6` as visible points in the triangular region `0 <= y <= x <= 6`; consecutive fractions have determinant one.

In [7]:

def polygon_lattice_boundary_count(vertices):
    total=0
    for a,b in zip(vertices,np.roll(vertices,-1,axis=0)):
        dx,dy=np.abs(b-a).astype(int); total+=math.gcd(int(dx),int(dy))
    return total
def point_in_poly(point,vertices):
    x,y=point; inside=False; n=len(vertices)
    for i in range(n):
        x1,y1=vertices[i]; x2,y2=vertices[(i+1)%n]
        if (y1>y)!=(y2>y):
            if x < (x2-x1)*(y-y1)/(y2-y1)+x1: inside=not inside
    return inside
poly=np.array([[-1,0],[3,0],[5,2],[2,4],[-2,3]],dtype=int); area_poly=abs(signed_area(poly)); boundary_count=polygon_lattice_boundary_count(poly)
minx,miny=poly.min(axis=0)-1; maxx,maxy=poly.max(axis=0)+1; all_pts=[(x0,y0) for x0 in range(minx,maxx+1) for y0 in range(miny,maxy+1)]
boundary_pts=[]; interior_pts=[]
for pt in all_pts:
    p0=np.array(pt,float); on_boundary=False
    for a,b in zip(poly,np.roll(poly,-1,axis=0)):
        if abs(cross2(p0-a,b-a))<1e-9 and min(a[0],b[0])-1e-9<=p0[0]<=max(a[0],b[0])+1e-9 and min(a[1],b[1])-1e-9<=p0[1]<=max(a[1],b[1])+1e-9: on_boundary=True; break
    if on_boundary: boundary_pts.append(pt)
    elif point_in_poly(pt,poly): interior_pts.append(pt)
n=6; farey=sorted({Fraction(y,x) for x in range(1,n+1) for y in range(0,x+1) if math.gcd(x,y)==1}); farey_pts=[(f.denominator,f.numerator) for f in farey]
adj_det=[f0.denominator*f1.numerator-f0.numerator*f1.denominator for f0,f1 in zip(farey,farey[1:])]
fig,axes=plt.subplots(1,2,figsize=(12.8,5.7))
for ax in axes: set_equal_2d(ax)
ax=axes[0]; xs,ys=zip(*all_pts); ax.scatter(xs,ys,s=16,facecolors="none",edgecolors="#cbd5e1")
if interior_pts: ix,iy=zip(*interior_pts); ax.scatter(ix,iy,s=36,color="#059669",label="interior")
if boundary_pts: bx,by=zip(*boundary_pts); ax.scatter(bx,by,s=36,color="#dc2626",label="boundary")
ax.add_patch(Polygon(poly,closed=True,facecolor="#2563eb22",edgecolor="#2563eb",lw=2.2)); ax.set_title(f"Pick polygon: area={area_poly:g}, b={boundary_count}, c={len(interior_pts)}"); ax.legend(loc="upper left",frameon=False); ax.set_xlim(minx-.2,maxx+.2); ax.set_ylim(miny-.2,maxy+.2)
ax=axes[1]
for x0 in range(0,n+1):
    for y0 in range(0,n+1):
        if 0<=y0<=x0<=n:
            primitive=(x0,y0) in farey_pts; ax.scatter([x0],[y0],s=44 if primitive else 18,color="#7c3aed" if primitive else "#cbd5e1")
for x0,y0 in farey_pts:
    ax.plot([0,x0],[0,y0],color="#7c3aed",alpha=.28,lw=1); ax.text(x0+.05,y0+.04,f"{y0}/{x0}",fontsize=7)
ax.plot([0,n,n,0],[0,0,n,0],color="#374151",lw=1.4); ax.set_title("Visible points for Farey sequence F6"); ax.set_xlim(-.3,n+.8); ax.set_ylim(-.3,n+.8)
fig.suptitle("Two-dimensional lattice invariants: area count and primitive visibility",y=1.02,fontsize=13); fig.tight_layout()
lattice_path=remember(FIG_DIR/"lattice-pick-farey-visibility.svg","two-dimensional lattices Pick Farey","figures"); fig.savefig(lattice_path,bbox_inches="tight"); plt.close(fig)
rows=[{"fraction":f"{f.numerator}/{f.denominator}","x":f.denominator,"y":f.numerator,"visible":True} for f in farey]; rows.append({"fraction":"PICK","x":int(boundary_count),"y":int(len(interior_pts)),"visible":True})
pick_csv=write_csv(DATA_DIR/"pick-farey-data.csv",rows); remember(pick_csv,"two-dimensional lattices Pick Farey","data")
checks={"concept":"two-dimensional lattices Pick Farey","polygon_vertices":poly.tolist(),"shoelace_area":float(area_poly),"boundary_lattice_points":int(boundary_count),"interior_lattice_points":int(len(interior_pts)),"pick_formula_value":float(boundary_count/2+len(interior_pts)-1),"pick_residual":float(area_poly-(boundary_count/2+len(interior_pts)-1)),"farey_order":n,"farey_sequence":[f"{f.numerator}/{f.denominator}" for f in farey],"adjacent_determinants":adj_det}
assert abs(checks["pick_residual"])<1e-12 and all(d==1 for d in adj_det)
write_check("lattice-pick-farey-checks.json",checks)
display_artifact(lattice_path,width=960)


## 5. Vectors, Centroids, and Barycentric Coordinates

A vector is the translation determined by an ordered pair of points. Once translations are written additively, affine coordinates become coefficients in a vector basis. Centroids are weighted averages that do not depend on which temporary origin was used to compute them. Barycentric coordinates extend the same idea to a triangle of reference: masses at the vertices determine a point, and normalized coordinates are the signed area shares of the opposite subtriangles.

In [8]:

E=np.array([1.35,.25]); F=np.array([.30,1.10]); origin=np.array([-.25,-.25]); coords=np.array([.70,.55]); P_vec=origin+coords[0]*E+coords[1]*F
tri=np.array([[.15,.10],[2.95,.35],[.75,2.35]]); weights=np.array([.22,.33,.45]); P_bar=weights@tri
area_coords=[]
for i in range(3): sub=tri.copy(); sub[i]=P_bar; area_coords.append(abs(signed_area(sub))/abs(signed_area(tri)))
area_coords=np.array(area_coords)
mass_pts=np.array([[-.8,0.],[.9,.3],[.2,1.35],[1.45,1.2]]); masses=np.array([2.,1.2,.8,-.35]); centroid=(masses[:,None]*mass_pts).sum(axis=0)/masses.sum(); other_origin=np.array([4.2,-2.1]); centroid_from_other=other_origin+(masses[:,None]*(mass_pts-other_origin)).sum(axis=0)/masses.sum()
fig,axes=plt.subplots(1,2,figsize=(12.8,5.6))
for ax in axes: set_equal_2d(ax)
ax=axes[0]; draw_arrow(ax,origin,E,color="#2563eb",label="e"); draw_arrow(ax,origin,F,color="#059669",label="f"); draw_arrow(ax,origin,P_vec-origin,color="#7c3aed",label="x e + y f")
ax.plot([origin[0]+E[0],origin[0]+E[0]+F[0],origin[0]+F[0]],[origin[1]+E[1],origin[1]+E[1]+F[1],origin[1]+F[1]],color="#d1d5db",ls="--")
ax.scatter(mass_pts[:,0],mass_pts[:,1],s=70*np.abs(masses),color="#f97316",alpha=.75,label="weighted points"); ax.scatter([centroid[0]],[centroid[1]],s=90,color="#111827",marker="x",label="centroid"); ax.text(centroid[0]+.06,centroid[1]+.06,"weighted centroid",fontsize=8); ax.set_title("Vectors and origin-independent centroid"); ax.legend(loc="upper left",frameon=False); ax.set_xlim(-1.2,2.3); ax.set_ylim(-.7,2)
ax=axes[1]; colors=["#2563eb","#059669","#dc2626"]
for i,color in enumerate(colors):
    sub=tri.copy(); sub[i]=P_bar; ax.add_patch(Polygon(sub,closed=True,facecolor=color+"33",edgecolor=color,lw=1.5,label=f"t{i+1}={weights[i]:.2f}"))
ax.add_patch(Polygon(tri,closed=True,fill=False,edgecolor="#111827",lw=2))
for i,pt in enumerate(tri,start=1): ax.scatter([pt[0]],[pt[1]],color="#111827",s=35); ax.text(pt[0]+.05,pt[1]+.05,f"A{i}",fontsize=9)
ax.scatter([P_bar[0]],[P_bar[1]],color="#111827",s=55); ax.text(P_bar[0]+.05,P_bar[1]+.05,"P",fontsize=9)
for pt in tri: ax.plot([P_bar[0],pt[0]],[P_bar[1],pt[1]],color="#6b7280",lw=1)
ax.set_title("Barycentric coordinates as signed area shares"); ax.legend(loc="upper right",frameon=False); ax.set_xlim(-.1,3.25); ax.set_ylim(-.05,2.65)
fig.suptitle("Vectors, centroids, and barycentric coordinates are one affine averaging language",y=1.02,fontsize=13); fig.tight_layout()
bary_path=remember(FIG_DIR/"vectors-centroids-barycentric.svg","vectors centroids barycentric coordinates","figures"); fig.savefig(bary_path,bbox_inches="tight"); plt.close(fig)
lam,mu_s,nu=sp.symbols("lambda mu nu"); A_b=sp.Matrix([1,0,0]); B_b=sp.Matrix([0,1,0]); C_b=sp.Matrix([0,0,1]); L_b=sp.Matrix([0,1,lam]); M_b=sp.Matrix([mu_s,0,1]); N_b=sp.Matrix([1,nu,0])
line_AL=A_b.cross(L_b); line_BM=B_b.cross(M_b); line_CN=C_b.cross(N_b); ceva_residual=sp.factor(line_CN.dot(line_AL.cross(line_BM))); menelaus_det=sp.factor(sp.Matrix.hstack(L_b,M_b,N_b).det())
bary_rows=[{"point":name,"t1":float(coord[0]),"t2":float(coord[1]),"t3":float(coord[2]),"sum":float(coord.sum())} for name,coord in zip(["A1","A2","A3","P"],[np.array([1,0,0]),np.array([0,1,0]),np.array([0,0,1]),weights])]
bary_csv=write_csv(DATA_DIR/"barycentric-samples.csv",bary_rows); remember(bary_csv,"vectors centroids barycentric coordinates","data")
checks={"concept":"vectors centroids barycentric coordinates","vector_coordinates":coords.tolist(),"centroid_origin_independence_error":float(np.linalg.norm(centroid-centroid_from_other)),"barycentric_weights":weights.tolist(),"barycentric_sum":float(weights.sum()),"area_coordinate_error":float(np.linalg.norm(area_coords-weights)),"point_reconstruction_error":float(np.linalg.norm(weights@tri-P_bar)),"ceva_symbolic_residual":str(ceva_residual),"ceva_condition":"lambda*mu*nu - 1 = 0","menelaus_symbolic_determinant":str(menelaus_det),"menelaus_condition":"lambda*mu*nu + 1 = 0"}
assert checks["centroid_origin_independence_error"]<1e-12 and abs(checks["barycentric_sum"]-1)<1e-12 and checks["area_coordinate_error"]<1e-12
assert sp.factor(ceva_residual + lam*mu_s*nu - 1)==0 and sp.factor(menelaus_det)==lam*mu_s*nu+1
write_check("barycentric-centroid-ceva-checks.json",checks)
display_artifact(bary_path,width=960)


## 6. Affine Space and Three-Dimensional Lattices

In three dimensions the same affine ideas use planes as well as lines. A basis of three independent translations builds a parallelepiped unit cell. A different triad gives the same lattice exactly when the integer change-of-basis matrix has determinant `+1` or `-1`; this is the 3D version of preserving unit cell volume.

The interactive artifact displays a slanted 3D lattice, one unit cell, a unimodular alternative cell, and a family of rational planes `X x + Y y + Z z = N` in lattice coordinates. The first rational planes, with `N = +/-1`, contain only visible lattice points because any common divisor of `(x, y, z)` would also divide `1`.

In [9]:

Bmat=np.array([[1.00,.35,.20],[0.,1.05,.28],[0.,.15,1.15]])
U=np.array([[1,1,0],[0,1,1],[0,0,1]],dtype=int); alt_B=Bmat@U.T; volume=abs(float(np.linalg.det(Bmat))); alt_volume=abs(float(np.linalg.det(alt_B)))
coords3=np.array(list(product(range(-1,3),repeat=3)),dtype=int); world=coords3@Bmat.T; coeff=np.array([1,2,1],dtype=int); Ns=[-1,0,1,2]
plane_rows=[]
for c in coords3:
    val=int(coeff@c)
    if val in Ns:
        plane_rows.append({"x":int(c[0]),"y":int(c[1]),"z":int(c[2]),"N":val,"visible":math.gcd(math.gcd(abs(int(c[0])),abs(int(c[1]))),abs(int(c[2])))==1})
plane_csv=write_csv(DATA_DIR/"three-dimensional-lattice-plane-points.csv",plane_rows); remember(plane_csv,"three-dimensional lattices rational planes","data")
fig=go.Figure(); fig.add_trace(go.Scatter3d(x=world[:,0],y=world[:,1],z=world[:,2],mode="markers",marker=dict(size=4,color="#64748b",opacity=.65),name="lattice points"))
def add_cell_edges(basis,name,color,dash=None):
    corners=np.array(list(product([0,1],repeat=3)),dtype=float); pts=corners@basis.T; edge_pairs=[]
    for i,a in enumerate(corners):
        for j,b in enumerate(corners):
            if j>i and np.sum(np.abs(a-b))==1: edge_pairs.append((i,j))
    for idx,(i,j) in enumerate(edge_pairs): fig.add_trace(go.Scatter3d(x=[pts[i,0],pts[j,0]],y=[pts[i,1],pts[j,1]],z=[pts[i,2],pts[j,2]],mode="lines",line=dict(color=color,width=5,dash=dash),name=name if idx==0 else None,showlegend=(idx==0)))
add_cell_edges(Bmat,"unit cell","#2563eb"); add_cell_edges(alt_B,"unimodular cell","#dc2626",dash="dash")
plane_colors={-1:"rgba(124,58,237,0.18)",0:"rgba(5,150,105,0.16)",1:"rgba(245,158,11,0.20)",2:"rgba(220,38,38,0.14)"}
for N in Ns:
    samples=[]
    for x0,y0 in [(-1.2,-1.2),(2.2,-1.2),(2.2,2.2),(-1.2,2.2)]:
        z0=(N-coeff[0]*x0-coeff[1]*y0)/coeff[2]; samples.append([x0,y0,z0])
    patch=np.array(samples)@Bmat.T; fig.add_trace(go.Mesh3d(x=patch[:,0],y=patch[:,1],z=patch[:,2],i=[0,0],j=[1,2],k=[2,3],opacity=.35,color=plane_colors[N],name=f"plane N={N}",showscale=False))
for vec,name,color in [(Bmat[:,0],"e","#2563eb"),(Bmat[:,1],"f","#059669"),(Bmat[:,2],"g","#7c3aed")]: fig.add_trace(go.Scatter3d(x=[0,vec[0]],y=[0,vec[1]],z=[0,vec[2]],mode="lines+text",line=dict(color=color,width=7),text=["",name],textposition="top center",name=f"basis {name}"))
fig.update_layout(title="3D affine lattice: unit cells, unimodular basis change, and rational planes",width=880,height=680,scene=dict(aspectmode="data",xaxis_title="X",yaxis_title="Y",zaxis_title="Z"),legend=dict(x=.02,y=.98))
lattice3d_path=remember(HTML_DIR/"three-dimensional-lattice-rational-planes.html","three-dimensional lattices rational planes","html"); fig.write_html(lattice3d_path,include_plotlyjs=True,full_html=True)
first_plane_points=[row for row in plane_rows if abs(row["N"])==1]
checks={"concept":"three-dimensional lattices rational planes","basis_matrix":Bmat.tolist(),"unit_cell_volume":volume,"unimodular_matrix":U.tolist(),"unimodular_determinant":int(round(np.linalg.det(U))),"alternative_cell_volume":alt_volume,"volume_difference":abs(volume-alt_volume),"rational_plane_coefficients":coeff.tolist(),"coefficients_gcd":math.gcd(math.gcd(abs(int(coeff[0])),abs(int(coeff[1]))),abs(int(coeff[2]))),"plane_rows_recorded":len(plane_rows),"all_recorded_plane_residuals_zero":all(int(coeff@np.array([row["x"],row["y"],row["z"]]))==row["N"] for row in plane_rows),"first_rational_plane_points_visible":all(row["visible"] for row in first_plane_points)}
assert abs(checks["unimodular_determinant"])==1 and checks["volume_difference"]<1e-12 and checks["coefficients_gcd"]==1 and checks["all_recorded_plane_residuals_zero"] and checks["first_rational_plane_points_visible"]
write_check("three-dimensional-lattice-checks.json",checks)
display_artifact(lattice3d_path,width=900)


## Takeaways

- Affine geometry keeps incidence, betweenness, parallelism, ratios on parallel lines, and signed area or volume ratios; it does not keep Euclidean length, angle, or circularity.
- The Desargues axiom is the engine behind well-defined dilatations: corresponding side pairs being parallel forces connector lines into either one center or one direction.
- Translations form an Abelian vector group. Half-turns and dilatations explain midpoints, parallelograms, and vector addition without measuring angles.
- Affinities are invertible line-preserving maps. Equiaffinities are the determinant-one affinities, so their natural invariant is area rather than shape.
- Lattices are affine objects once they are described by integer coordinate spans. Primitive or visible points, Pick's theorem, Farey adjacency, and unimodular basis changes are determinant statements.
- Barycentric coordinates are homogeneous masses. After normalization they become areal coordinates, and determinant tests give linear equations for lines plus Ceva and Menelaus conditions.
- In 3D, the same vector and determinant language builds affine space, parallelepiped unit cells, rational planes, and first rational plane visibility.

In [10]:

manifest_path=write_csv(TABLE_DIR/"artifact_manifest.csv",ARTIFACTS_WRITTEN); remember(manifest_path,"chapter 13 artifact manifest","tables")
artifact_paths=[BOOK_ROOT/row["path"] for row in ARTIFACTS_WRITTEN]
assert_artifacts(artifact_paths,min_bytes=80)
check_payloads={}
for check_path in CHECK_FILES: check_payloads[check_path.name]=json.loads(check_path.read_text(encoding="utf-8"))
required_checks=["source-span.json","parallelism-desargues-checks.json","desargues-proof-dependency-checks.json","dilatation-group-checks.json","affinity-area-symbolic-checks.json","equiaffine-shear-lab-checks.json","lattice-pick-farey-checks.json","barycentric-centroid-ceva-checks.json","three-dimensional-lattice-checks.json"]
missing=[name for name in required_checks if name not in check_payloads]; assert not missing,missing
assert check_payloads["parallelism-desargues-checks.json"]["max_abs_residual"]<1e-10
assert check_payloads["dilatation-group-checks.json"]["halfturn_product_error"]<1e-10
assert check_payloads["affinity-area-symbolic-checks.json"]["hyperbolic_xy_invariant_symbolic"]=="0"
assert abs(check_payloads["lattice-pick-farey-checks.json"]["pick_residual"])<1e-12
assert check_payloads["barycentric-centroid-ceva-checks.json"]["ceva_symbolic_residual"] in {"lambda*mu*nu - 1","-lambda*mu*nu + 1"}
assert abs(check_payloads["three-dimensional-lattice-checks.json"]["unimodular_determinant"])==1
visual_summary={"concept":"chapter 13 final sanity summary","chapter":CHAPTER_NO,"source_span":source_span,"artifact_count":len(ARTIFACTS_WRITTEN),"check_count":len(CHECK_FILES),"artifacts":ARTIFACTS_WRITTEN,"required_checks":required_checks,"all_artifacts_nonempty":all(path.exists() and path.stat().st_size>80 for path in artifact_paths)}
summary_path=write_check("visual_summary.json",visual_summary)
assert summary_path.exists() and summary_path.stat().st_size>80
assert_artifacts([summary_path,manifest_path],min_bytes=80)
display(Markdown(f"Final sanity passed: {len(ARTIFACTS_WRITTEN)} artifacts tracked, {len(CHECK_FILES)} check files written."))
visual_summary


Final sanity passed: 23 artifacts tracked, 10 check files written.

{'concept': 'chapter 13 final sanity summary',
 'chapter': 13,
 'source_span': {'book': 'Introduction to Geometry',
  'chapter': 13,
  'printed_pages': '191-228',
  'pdf_pages': '209-246',
  'source_map': 'pdf_page = printed_page + 18',
  'inspected': True},
 'artifact_count': 22,
 'check_count': 9,
 'artifacts': [{'concept': 'source span orientation',
   'kind': 'checks',
   'path': 'artifacts/chapter-13/checks/source-span.json'},
  {'concept': 'parallelism and Desargues axiom',
   'kind': 'figures',
   'path': 'artifacts/chapter-13/figures/parallelism-desargues-configuration.svg'},
  {'concept': 'parallelism and Desargues axiom',
   'kind': 'checks',
   'path': 'artifacts/chapter-13/checks/parallelism-desargues-checks.json'},
  {'concept': 'proof dependency scaffold',
   'kind': 'figures',
   'path': 'artifacts/chapter-13/figures/desargues-proof-dependency.svg'},
  {'concept': 'proof dependency scaffold',
   'kind': 'checks',
   'path': 'artifacts/chapter-13/checks/desargues-proof-de